# Multi-Agent News Brief

Find news, filter it with an editor, write a script, and generate a voice briefing with OpenAI or Gemini.

## 1. Install dependencies

In [8]:
%pip install -q openai-agents ddgs google-genai

## 2. Select an agent provider

Set `PROVIDER` to `"openai"` or `"gemini"`. The same provider also generates the final voice briefing.

In [9]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "openai"  # Change to "gemini" to use Gemini for agents and TTS.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

if PROVIDER == "openai":
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets for Gemini agents.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Find recent news

In [10]:
from ddgs import DDGS


def search_news(query: str) -> str:
    """Find recent news and return each item's title, summary, date, and URL."""
    results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    if not results:
        raise RuntimeError("No recent news results were found. Try a different topic.")

    for index, item in enumerate(results, start=1):
        print(f"{index}. {item.get('title', 'Untitled')}")
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\nDate: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\nURL: {item.get('url', '')}"
        for item in results
    )


topic = "artificial intelligence business"
news_items = search_news(topic)

1. As banks' AI use evolves, core truths about the business still apply
https://www.americanbanker.com/opinion/as-banks-ai-use-evolves-core-truths-about-the-business-still-apply
2. Vizrt Named a Leader in the 2026 IDC MarketScape for Worldwide AI-Driven Business Cycle in the Media Industry
https://finance.yahoo.com/media-advertising/articles/vizrt-named-leader-2026-idc-070000613.html
3. Cyprus channels €500 million into business modernisation
https://cyprus-mail.com/2026/09/10/cyprus-channels-e500-million-into-business-modernisation
4. BRICS business: Jaishankar unveils India's economic vision at BRICS forum
https://www.msn.com/en-in/money/economy/brics-business-jaishankar-unveils-india-s-economic-vision-at-brics-forum/vi-AA2c0kuk
5. Diamond Podcast: Emotional Intelligence as an Advisor Edge
https://www.wealthmanagement.com/client-relations/the-diamond-podcast-for-financial-advisors-emotional-intelligence-is-advisors-advantage-in-an-ai-world
6. Trump admin partners with OpenAI to equip

## 4. Editor Agent: filter and refine

In [11]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="Select the three most relevant, credible, non-duplicative items. Exclude ads, clickbait, speculation, and weak evidence. Preserve each selected item's key facts, date, and URL.",
    model=model,
)

editor_result = await Runner.run(editor_agent, f"Topic: {topic}\n\nNews items:\n{news_items}")
edited_news = editor_result.final_output
print(edited_news)

- As banks' AI use evolves, core truths about the business still apply — 2026-09-11. Key facts: Opinion on how AI will change banking operations but won't replace fundamentals like care for employees, customers and shareholders. URL: https://www.americanbanker.com/opinion/as-banks-ai-use-evolves-core-truths-about-the-business-still-apply

- Trump admin partners with OpenAI to equip federal employees with artificial intelligence tools — 2026-09-10. Key facts: GSA secured a 50% discount on OpenAI's AI models for federal employees, with no minimum spend required. URL: https://www.msn.com/en-us/news/other/trump-admin-partners-with-openai-to-equip-federal-employees-with-artificial-intelligence-tools/ar-AA2bXHXj

- OpenAI and Anthropic top our list of private companies, but their reign will be short — 2026-09-11. Key facts: BizJournals reports OpenAI and Anthropic led a list of private Bay Area companies, collectively pulling in $23 billion in revenue last year per the Business Times list. U

## 5. Writer Agent: create the script

In [12]:
writer_agent = Agent(
    name="News Script Writer",
    instructions="Write a neutral 45- to 60-second spoken news script using only the editor's selected items. Do not add unsupported facts. End by naming the source publications without reading URLs aloud.",
    model=model,
)

writer_result = await Runner.run(writer_agent, f"Create a spoken news script from this edited brief:\n\n{edited_news}")
news_script = writer_result.final_output
print(news_script)

In business headlines: An opinion in American Banker on Sept. 11 argues that as banks' use of artificial intelligence evolves, core truths about the business still apply. The piece says AI will change banking operations but will not replace fundamentals like care for employees, customers and shareholders.

MSN reports on Sept. 10 that the Trump administration has partnered with OpenAI to equip federal employees with AI tools. The General Services Administration secured a 50 percent discount on OpenAI's models for federal employees, with no minimum spend required.

And BizJournals on Sept. 11 says OpenAI and Anthropic topped a list of private Bay Area companies, collectively pulling in $23 billion in revenue last year, though the outlet says their reign may be short.

Sources: American Banker, MSN, and BizJournals.


## 6. Generate the voice briefing

This creates AI-generated speech with the selected provider. Disclose that the voice is AI-generated when sharing it.

In [13]:
from IPython.display import Audio, display

if PROVIDER == "openai":
    tts_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    speech = await tts_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input=news_script,
        instructions="Speak clearly in a calm, neutral broadcast-news style.",
    )
    audio_path = "news_brief.mp3"
    await speech.write_to_file(audio_path)
else:
    import base64
    import wave

    from google import genai

    def save_wav(filename, pcm, channels=1, rate=24000, sample_width=2):
        with wave.open(filename, "wb") as wav_file:
            wav_file.setnchannels(channels)
            wav_file.setsampwidth(sample_width)
            wav_file.setframerate(rate)
            wav_file.writeframes(pcm)

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    response = gemini_client.interactions.create(
        model="gemini-3.1-flash-tts-preview",
        input=(
            "Read this in a calm, neutral broadcast-news style:\n\n"
            f"{news_script}"
        ),
        response_format={"type": "audio"},
        generation_config={"speech_config": [{"voice": "Kore"}]},
    )
    audio_path = "news_brief.wav"
    save_wav(audio_path, base64.b64decode(response.output_audio.data))

display(Audio(audio_path))


TypeError: object NoneType can't be used in 'await' expression

## 7. Download the audio file

Run this cell to save the generated MP3 (OpenAI) or WAV (Gemini) file to your computer.

In [ ]:
from google.colab import files

files.download(audio_path)